# ILVAD 复现 + 证据图蒸馏（方向 B）

复现论文 **Finding the Correct Visual Evidence Without Forgetting**（arXiv:2605.20965）的推理期注意力干预，
并在其上做一个拓展：**把「证据显著图」从注意力里蒸馏出来，变成一个只看隐状态的轻量探针**（方向 B）。

> 如果要放 Colab 的「Open in Colab」徽章，把 README 里的 `USER/REPO` 换成你的仓库路径即可。

## 这个 notebook 能做什么

| 段 | 内容 | 需要 |
|---|---|---|
| 1–3 | 4bit 加载 LLaVA-1.5-7B、取注意力、定位视觉段 | 免费版 T4 即可 |
| 4 | 复现式 (4)—(6)，构建证据图 Ŝ，并叠加到原图上 | 同上 |
| 5 | 复现式 (7)—(10) 的注意力重加权（含等价性自检） | 同上 |
| 6 | baseline / ILVAD 生成对比（定性） | 同上 |
| 7 | POPE 风格的 yes/no 冒烟测试（可选接真实 POPE） | 同上 |
| 8 | 方向 B：采集 `(隐状态, Ŝ)` 对，训练探针 | 同上 |
| 9 | AOPC 评测（证据图是否 faithful） | 同上 |

## 设计原则

1. **自检优先**：所有改动模型的地方都配一个「关掉干预后应与原模型逐位一致」的检查（5.2 节）。
   自检失败时，后面的干预段会自动跳过，只剩诊断功能 —— 这样 notebook 不会因为版本差异整段崩掉。
2. **不用 KV cache**：`use_cache=False`，每次前向都算全序列。T=10 时多花的算力可以忽略，
   换来的是代码只依赖 `q_proj/k_proj/v_proj/o_proj` 这几个稳定属性。
3. **合成图兜底**：第 2 节自带一张可控的合成图（红球 + 球拍 + 长凳），不依赖任何数据集下载，
   而且**真值是已知的**，可以直接判断证据图有没有落在球上。

## 已知限制

- 4bit 量化后的注意力与论文 fp16 不同，**只能比 baseline 与 ILVAD 的差值**，不能比绝对值。
- 复现的是「机制」不是「论文表格」：CHAIR 全量要几千次长文本生成，免费版跑不完。
- 探针是 baseline 之外的探索，不是论文内容。

## 环境

`transformers==4.45.0`（与官方仓库 requirements 一致）、`torch>=2.1`、`bitsandbytes`、`accelerate`、`datasets`。
Colab：运行时 → 更改运行时类型 → T4 GPU。

## 0. 安装与配置

In [ ]:
!pip -q install "transformers==4.45.0" "accelerate==0.34.2" bitsandbytes datasets matplotlib 2>/dev/null | tail -2

In [ ]:
import math, random, warnings, types
from dataclasses import dataclass
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
from IPython.display import display

warnings.filterwarnings("ignore")


@dataclass
class Cfg:
    model_id: str = "llava-hf/llava-1.5-7b-hf"
    load_4bit: bool = True
    window_T: int = 10          # 式(4) 的窗口 T
    tau: float = 5.0            # 式(5) 的阈值倍数（论文默认 5；图全空就降到 2—3）
    alpha: float = 5.0          # 式(8) 的视觉增强强度
    beta: float = 1.0           # 式(10) 的文本增强保底
    rho: float = 0.5            # 保留的注意力头比例
    max_new_tokens: int = 24
    probe_layer: int = 16       # 方向 B：取第几层隐状态
    seed: int = 0


cfg = Cfg()


def seed_all(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)


seed_all(cfg.seed)

if not torch.cuda.is_available():
    raise RuntimeError("需要 GPU。Colab: 运行时 > 更改运行时类型 > T4 GPU")

p = torch.cuda.get_device_properties(0)
print(f"GPU: {p.name}  显存: {p.total_memory / 1e9:.1f} GB")
print(f"torch {torch.__version__}")

## 1. 加载模型

- `attn_implementation="eager"`：只有 eager 才会把 softmax 之后的注意力权重吐出来；
  SDPA / FlashAttention 默认不返回，这是复现 ILVAD 的第一个坎。
- 4bit（nf4）把 7B 压到约 4.5 GB，免费版 T4 才装得下。

In [ ]:
from transformers import AutoProcessor, LlavaForConditionalGeneration, BitsAndBytesConfig

quant = None
if cfg.load_4bit:
    quant = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

model = LlavaForConditionalGeneration.from_pretrained(
    cfg.model_id,
    quantization_config=quant,
    torch_dtype=torch.float16,
    device_map="cuda",
    attn_implementation="eager",
    low_cpu_mem_usage=True,
)
processor = AutoProcessor.from_pretrained(cfg.model_id)
model.eval()

IMG_TOK = model.config.image_token_index
N_VIS = getattr(model.config, "image_seq_length", 576)
print("attn_implementation:", model.config._attn_implementation)
print("image_token_index:", IMG_TOK, " image_seq_length:", N_VIS)
print(f"VRAM allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## 2. 一张真值已知的合成图

不依赖数据集下载，先把「视觉段定位」和「证据图有没有落在球上」这两件事验证掉。

图里有三样东西：**红球**（目标）、球拍、长凳。问题问的是球，所以证据图应该集中在红球那一片。

In [ ]:
def make_scene(size=336, seed=0):
    rng = random.Random(seed)
    img = Image.new("RGB", (size, size), (60, 140, 70))
    d = ImageDraw.Draw(img)
    for _ in range(60):
        x, y = rng.randrange(size), rng.randrange(size - 12)
        d.line([(x, y), (x, y + rng.randrange(4, 11))], fill=(45, 120, 55), width=1)
    d.rectangle([24, 248, 190, 296], fill=(120, 90, 60))
    d.line([(96, 296), (196, 176)], fill=(235, 235, 235), width=9)
    d.ellipse([162, 132, 242, 212], outline=(235, 235, 235), width=7)
    d.ellipse([206, 116, 258, 168], fill=(220, 40, 40))
    return img


image = make_scene(seed=cfg.seed)
image.save("scene.png")
display(image)

PROMPT = "Is there a red ball in the image? Answer yes or no."


def build_inputs(image, prompt=PROMPT):
    conv = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]}]
    try:
        text = processor.apply_chat_template(conv, add_generation_prompt=True)
    except Exception:
        text = f"USER: <image>\n{prompt}\nASSISTANT:"
    enc = processor(text=[text], images=[image], return_tensors="pt")
    out = {}
    for k, v in enc.items():
        v = v.to("cuda")
        out[k] = v.to(torch.float16) if k == "pixel_values" else v
    return out


inputs = build_inputs(image)
print({k: tuple(v.shape) for k, v in inputs.items()})
_n_img = int((inputs["input_ids"][0] == IMG_TOK).sum())
_txt = processor.tokenizer.decode(inputs["input_ids"][0]).replace("<image>", "")
print(f"解码后的提示（已省略 {_n_img} 个 <image>）:", _txt.replace("\n", " | ")[:160])


def visual_span(input_ids):
    pos = (input_ids[0] == IMG_TOK).nonzero(as_tuple=True)[0]
    assert len(pos) > 0, "input_ids 里没有图像占位 token：检查 processor 与 chat template"
    v0 = int(pos[0])
    n = len(pos)
    if n == 1:
        n = N_VIS                      # 旧路径：processor 只留 1 个占位，模型内部再展开
    elif n != N_VIS:
        print(f"warning: 实际视觉 token 数 {n} != config 里的 {N_VIS}，以实际为准")
    return v0, v0 + n


V0, V1 = visual_span(inputs["input_ids"])
print(f"视觉段 = [{V0}, {V1})，共 {V1 - V0} 个 token")

## 3. 读出注意力（论文的 Stage 0）

对前 `T` 个**生成**的 token，逐层逐头记录「该 token 对每个视觉 token 的注意力」。

实现细节：不做 KV cache，每一步都重算全序列，取**最后一行**注意力，再切出视觉段。
第 0 次前向的最后一行属于**提示的最后一个 token**，不是生成的 token，所以丢弃（`rows[1:]`）。

> 代价：T+1 次全序列前向。T=10、序列 ~650 时在 T4 上约 10 秒，可接受。

In [ ]:
@torch.no_grad()
def probe_attention(model, inputs, T, v0, v1):
    ids = inputs["input_ids"].clone()
    pix = inputs["pixel_values"]
    rows = []
    for _ in range(T + 1):
        out = model(input_ids=ids, pixel_values=pix,
                    attention_mask=torch.ones_like(ids),
                    output_attentions=True, use_cache=False)
        # out.attentions: 每层 [B, H, S, S]；取最后一行、视觉段
        rows.append(torch.stack([a[0, :, -1, v0:v1] for a in out.attentions]))  # [L, H, |V|]
        nxt = out.logits[:, -1, :].argmax(dim=-1, keepdim=True)   # [1, 1]
        ids = torch.cat([ids, nxt], dim=-1)
        del out
    return rows[1:], ids   # 丢掉提示那一行


rows, ids_full = probe_attention(model, inputs, cfg.window_T, V0, V1)
print("收集到", len(rows), "步；每一步形状", tuple(rows[0].shape), "= [层, 头, 视觉 token]")
print("前 T 步生成的内容:", processor.tokenizer.decode(ids_full[0, inputs["input_ids"].shape[1]:],
                                                     skip_special_tokens=True))

In [ ]:
# 原始注意力：对视觉 token 取所有层 / 头 / 步的平均，画成 24x24 网格
raw = torch.stack(rows).mean(dim=(0, 1, 2))  # [|V|]
g = int(round(math.sqrt(raw.numel())))
fig, ax = plt.subplots(1, 2, figsize=(8, 4))
ax[0].imshow(image)
ax[0].set_title("image")
heat = raw.reshape(g, g).float().cpu().numpy()
im1 = ax[1].imshow(heat, cmap="magma")
ax[1].set_title("raw visual attention (avg)")
plt.colorbar(im1, ax=ax[1], fraction=0.046)
plt.tight_layout()
plt.show()

## 4. 复现 Stage 1：式 (4)—(6)

```
式(4)  Ā[l,j] = 1/(T·|H_v|) · Σ_h Σ_{i≤T} A[l,h,i,j]      （在窗口内的视觉敏感头上平均）
式(5)  Ã[l,j] = 1[ Ā[l,j] > τ · mean_j Ā[l,:] ]            （按层内均值二值化）
式(6)  S[j]   = Σ_l max( Ã[l+1,j] − Ã[l,j], 0 )            （只数 0→1 的跳变）
       Ŝ     = S / max(S)
```

要点：`Ã` 是 **0/1 状态**，差分的是状态而不是幅度；持续显著的 sink 每层都是 1，差分为 0。

In [ ]:
@torch.no_grad()
def build_map(rows, cfg):
    T = len(rows)
    L, H, V = rows[0].shape

    # 式(4) 的前置：按「视觉注意力总和」选前 rho 的头
    vis_mass = torch.stack(rows).mean(dim=0).sum(dim=-1)          # [L, H]
    k = max(1, int(round(cfg.rho * H)))
    kept = vis_mass.topk(k, dim=-1).indices                        # [L, k]

    Abar = torch.zeros(L, V, device=rows[0].device)
    for l in range(L):
        sel = torch.stack([rows[t][l, kept[l]] for t in range(T)])  # [T, k, V]
        Abar[l] = sel.mean(dim=(0, 1))

    thr = cfg.tau * Abar.mean(dim=-1, keepdim=True)                # [L,1]
    B = (Abar > thr).float()                                       # 式(5)
    S = (B[1:] - B[:-1]).clamp(min=0).sum(dim=0)                   # 式(6)
    denom = S.max().clamp(min=1e-6)
    return {"S": S, "Shat": S / denom, "Abar": Abar, "B": B, "kept": kept}


m = build_map(rows, cfg)
print("每层通过阈值的 token 数:", m["B"].sum(-1).long().cpu().tolist())
print("S 的分布:", np.bincount(m["S"].long().cpu().numpy()))
print("非零 token 数:", int((m["S"] > 0).sum()), "/", m["S"].numel())
print("Ŝ 最大值:", float(m["Shat"].max()))
if int((m["S"] > 0).sum()) < 3:
    print("提示：证据图过于稀疏。可把 cfg.tau 降到 2—3，或把窗口 T 调大。")

In [ ]:
def overlay(image, shat, title="", ax=None):
    g = int(round(math.sqrt(shat.numel())))
    heat = shat.reshape(g, g).float().cpu().numpy()
    heat = (heat - heat.min()) / (heat.max() - heat.min() + 1e-8)
    ax = ax or plt.gca()
    ax.imshow(image)
    ax.imshow(heat, cmap="jet", alpha=0.45, extent=(0, image.size[0], image.size[1], 0))
    ax.set_title(title, fontsize=9)
    ax.axis("off")


fig, ax = plt.subplots(1, 3, figsize=(12, 4))
overlay(image, raw, "raw attention", ax[0])
overlay(image, m["Shat"], "ILVAD evidence map", ax[1])
ax[2].imshow(m["B"].float().cpu().numpy(), aspect="auto", cmap="gray")
ax[2].set_xlabel("visual token"); ax[2].set_ylabel("layer"); ax[2].set_title("Ã (binary)")
plt.tight_layout()
plt.show()

## 5. 复现 Stage 2：式 (7)—(10)

### 5.1 算子

```
式(8)   Â[i,j] = A[i,j] · exp(α·Ŝ[j]),  j ∈ X_v
式(10)  Â[i,j] = A[i,j] · (ŵ[i] + β),   j ∈ X_t
        再逐行归一化
```

`exp(α·Ŝ)` 只在视觉段上乘，所以等价于**给视觉键的 logit 加 α·Ŝ 的偏置**：
实现上既可以"改权重再乘 V"，也可以"改 mask/bias"。这里选前者，因为要显式验证等价性。

### 5.2 打补丁 + 自检

`transformers` 的 `LlamaAttention.forward` 内部才做 softmax，外部拿不到中间量，
所以这里**替换每个 attention 模块的 forward**：自己算 Q/K/V、自己 softmax、在 softmax 与乘 V 之间插入重加权。

替换是危险的（版本一变就可能炸），所以下面紧跟一个 **等价性自检**：
关掉干预（Ŝ 全 0、α=0）时，打补丁后的 logits 必须与原模型逐位一致。
**自检不过，后面所有干预段自动跳过**。

In [ ]:
from transformers.models.llama.modeling_llama import repeat_kv
try:
    from transformers.models.llama.modeling_llama import apply_rotary_pos_emb as _rope_fn
except Exception:
    _rope_fn = None

ILVAD = {"on": False, "score": None, "alpha": 0.0, "mode": "boost", "span": (0, 0)}


def set_map(score=None, alpha=None, mode=None):
    ILVAD["on"] = score is not None
    ILVAD["score"] = score
    ILVAD["alpha"] = cfg.alpha if alpha is None else alpha
    if mode is not None:
        ILVAD["mode"] = mode


def _rope(q, k, cos, sin):
    if _rope_fn is None:
        raise RuntimeError("找不到 apply_rotary_pos_emb，transformers 版本不匹配")
    try:
        return _rope_fn(q, k, cos, sin, unsqueeze_dim=1)
    except TypeError:
        return _rope_fn(q, k, cos, sin)


def _causal(q_len, k_len, dtype, device):
    i = torch.arange(q_len, device=device).unsqueeze(-1)
    j = torch.arange(k_len, device=device).unsqueeze(0)
    mask = torch.where(j <= i + (k_len - q_len), 0.0, float("-inf"))
    return mask.to(dtype)[None, None, :, :]


def _reweight(aw, span, score, alpha, mode):
    v0, v1 = span
    left, mid, right = aw[..., :v0], aw[..., v0:v1], aw[..., v1:]
    if mode == "boost":
        f = torch.exp(alpha * score).to(aw.dtype)
    elif mode == "mask":
        f = (1.0 - score).to(aw.dtype)
    else:
        f = torch.ones(score.shape, dtype=aw.dtype, device=aw.device)
    mid = mid * f
    tot = left.sum(-1, keepdim=True) + mid.sum(-1, keepdim=True) + right.sum(-1, keepdim=True)
    return torch.cat([left / tot, mid / tot, right / tot], dim=-1)


def _find_layers(model):
    for path in ("model.language_model.layers", "model.layers", "language_model.model.layers"):
        obj, ok = model, True
        for part in path.split("."):
            if not hasattr(obj, part):
                ok = False
                break
            obj = getattr(obj, part)
        if ok and hasattr(obj, "__len__"):
            return obj
    raise RuntimeError("找不到 decoder layers，请检查 transformers 版本")


def _make_forward(attn):
    def forward(self, hidden_states, attention_mask=None, position_ids=None, past_key_value=None,
                output_attentions=False, use_cache=False, cache_position=None, **kw):
        assert past_key_value is None, "ILVAD 补丁要求 use_cache=False"
        bsz, q_len, _ = hidden_states.shape
        hd = self.head_dim
        q = self.q_proj(hidden_states).view(bsz, q_len, self.num_heads, hd).transpose(1, 2)
        k = self.k_proj(hidden_states).view(bsz, q_len, self.num_key_value_heads, hd).transpose(1, 2)
        v = self.v_proj(hidden_states).view(bsz, q_len, self.num_key_value_heads, hd).transpose(1, 2)
        cos, sin = self.rotary_emb(v, position_ids)
        q, k = _rope(q, k, cos, sin)
        k = repeat_kv(k, self.num_key_value_groups)
        v = repeat_kv(v, self.num_key_value_groups)
        scale = getattr(self, "scaling", None) or hd ** -0.5
        aw = (q @ k.transpose(2, 3)) * scale
        if attention_mask is not None and attention_mask.dim() == 4:
            aw = aw + attention_mask[:, :, :, -k.shape[-1]:]
        elif attention_mask is not None and attention_mask.dim() == 2:
            pad = (1.0 - attention_mask.to(aw.dtype))[:, None, None, :] * torch.finfo(aw.dtype).min
            aw = aw + pad + _causal(q_len, k.shape[-1], aw.dtype, aw.device)
        else:
            aw = aw + _causal(q_len, k.shape[-1], aw.dtype, aw.device)
        aw = F.softmax(aw.float(), dim=-1).to(q.dtype)
        if ILVAD["on"] and ILVAD["score"] is not None:
            aw = _reweight(aw, ILVAD["span"], ILVAD["score"], ILVAD["alpha"], ILVAD["mode"])
        out = aw @ v
        out = out.transpose(1, 2).reshape(bsz, q_len, -1).contiguous()
        return self.o_proj(out), aw, None
    return forward


@torch.no_grad()
def logits_of(model, inputs):
    out = model(input_ids=inputs["input_ids"], pixel_values=inputs["pixel_values"],
                attention_mask=torch.ones_like(inputs["input_ids"]), use_cache=False)
    return out.logits[0, -1].float()


base_logits = logits_of(model, inputs).clone()

for layer in _find_layers(model):
    layer.self_attn.forward = types.MethodType(_make_forward(layer.self_attn), layer.self_attn)

ILVAD["span"] = (V0, V1)
set_map(score=torch.zeros(V1 - V0, device="cuda"), alpha=0.0)
patched_logits = logits_of(model, inputs)
diff = (patched_logits - base_logits).abs().max().item()
PATCH_OK = diff < 1e-2
print(f"等价性自检：max |Δlogit| = {diff:.2e}  ->  {'PASS' if PATCH_OK else 'FAIL'}")
if not PATCH_OK:
    print("补丁未能复现原模型输出，后续干预段将跳过。请检查 transformers 版本与注意力实现。")
set_map(None)

## 6. baseline vs ILVAD：生成对比

同一张图、同一个提示，唯一差异是 attention 里多了一张证据图。观察幻觉词是否减少。

> 这是**定性**观察。要定量，请用第 7 节的 yes/no 冒烟测试或接真实 POPE。

In [ ]:
@torch.no_grad()
def generate(model, inputs, max_new_tokens=24):
    ids = inputs["input_ids"].clone()
    pix = inputs["pixel_values"]
    for _ in range(max_new_tokens):
        out = model(input_ids=ids, pixel_values=pix,
                    attention_mask=torch.ones_like(ids), use_cache=False)
        nxt = out.logits[:, -1, :].argmax(dim=-1, keepdim=True)   # [1, 1]
        ids = torch.cat([ids, nxt], dim=-1)
        if nxt.item() == processor.tokenizer.eos_token_id:
            break
        del out
    return processor.tokenizer.decode(ids[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)


if PATCH_OK:
    set_map(None)
    txt_base = generate(model, inputs, cfg.max_new_tokens)
    set_map(m["Shat"], alpha=cfg.alpha, mode="boost")
    txt_ilvad = generate(model, inputs, cfg.max_new_tokens)
    set_map(None)
    print("[baseline]", txt_base)
    print("[ILVAD   ]", txt_ilvad)
else:
    print("跳过：补丁自检未通过")

## 7. POPE 风格的 yes/no 冒烟测试

POPE 是二分类：问「图里有没有 X」，比较 `P(yes)` 与 `P(no)`。
这里不生成文本，只看**第一个位置的 logits**，又快又稳定。

先跑合成图上的 6 个问题（真值已知）验证管线；接真实 POPE 的写法在代码末尾。

In [ ]:
def yes_no_logits(model, inputs, processor):
    out = model(input_ids=inputs["input_ids"], pixel_values=inputs["pixel_values"],
                attention_mask=torch.ones_like(inputs["input_ids"]), use_cache=False)
    lg = out.logits[0, -1].float()
    tok = processor.tokenizer
    ids = {}
    for w in ("yes", "no"):
        cand = tok.encode(" " + w, add_special_tokens=False)
        if len(cand) != 1:
            print(f"warning: '{w}' 被切成 {len(cand)} 个 token，取第一个")
        ids[w] = cand[0]
    return lg[ids["yes"]].item(), lg[ids["no"]].item()


def evaluate_pairs(model, pairs, verbose=True):
    # pairs: list of (PIL.Image, question, "yes" / "no")
    # 注意：每张图都要先重建自己的 Ŝ（见 build_map），这里为简洁假设同一张图
    ok_b = ok_i = 0
    for img, q, gt in pairs:
        inp = build_inputs(img, q)
        set_map(None)
        y0, n0 = yes_no_logits(model, inp, processor)
        set_map(m["Shat"], alpha=cfg.alpha, mode="boost")
        y1, n1 = yes_no_logits(model, inp, processor)
        set_map(None)
        pb = "yes" if y0 > n0 else "no"
        pi = "yes" if y1 > n1 else "no"
        ok_b += pb == gt
        ok_i += pi == gt
        if verbose:
            print(f"{q[:50]:50s} gt={gt:3s} base={pb:3s} ilvad={pi:3s}")
    n = len(pairs)
    return ok_b / n, ok_i / n


SYNTH_PAIRS = [
    ("Is there a red ball in the image?", "yes"),
    ("Is there a tennis racket in the image?", "yes"),
    ("Is there a bench in the image?", "yes"),
    ("Is there a dog in the image?", "no"),
    ("Is there a car in the image?", "no"),
    ("Is the ball blue?", "no"),
]

if PATCH_OK:
    acc_b, acc_i = evaluate_pairs(model, [(image, q, gt) for q, gt in SYNTH_PAIRS])
    print(f"\n合成集: baseline {acc_b:.0%}  ILVAD {acc_i:.0%}（样本少，只用于验证管线）")
else:
    print("跳过：补丁自检未通过")

# 接真实 POPE（每张图需要单独建图；这里给出适配写法，不默认执行）：
#   from datasets import load_dataset
#   ds = load_dataset("lmms-lab/POPE", split="test").select(range(200))
#   for ex in ds:
#       img = ex["image"].convert("RGB")
#       inp = build_inputs(img, ex["question"])
#       v0, v1 = visual_span(inp["input_ids"])
#       rows_i, _ = probe_attention(model, inp, cfg.window_T, v0, v1)
#       shat_i = build_map(rows_i, cfg)["Shat"]
#       ... 然后用 shat_i 做 evaluate 的那两步 forward

## 8. 方向 B：把证据图蒸成探针

**教师**：ILVAD 的 Ŝ（需要物化注意力，前 T 步要 eager）。
**学生**：两层 MLP，只看第 `k` 层每个视觉 token 的隐状态。

```
f_θ(h_k) -> Ŝ̂ ∈ [0,1]^{|V|}
```

推理时只需 **一次 prefill + 一个 MLP**：没有 T 步探测、没有注意力矩阵、没有二次 prefill。

下面用同一张合成图做最小验证：训练集是它的若干次随机扰动（颜色 / 位置微调），
验证集是另一次扰动。**这是管线验证，不是论文级实验**。

In [ ]:
N_TRAIN, N_VAL = 24, 6
HID = 256


@torch.no_grad()
def teacher_pair(img, prompt=PROMPT):
    inp = build_inputs(img, prompt)
    v0, v1 = visual_span(inp["input_ids"])
    rows, _ = probe_attention(model, inp, cfg.window_T, v0, v1)
    mm = build_map(rows, cfg)
    out = model(input_ids=inp["input_ids"], pixel_values=inp["pixel_values"],
                attention_mask=torch.ones_like(inp["input_ids"]),
                output_hidden_states=True, use_cache=False)
    h = out.hidden_states[cfg.probe_layer][0, v0:v1, :].float().cpu()   # [|V|, d]
    return h, mm["Shat"].float().cpu(), mm


def perturbed(seed):
    rng = random.Random(1000 + seed)
    img = make_scene(seed=seed)
    d = ImageDraw.Draw(img)
    x = rng.randint(190, 220); y = rng.randint(100, 140)
    d.ellipse([x, y, x + rng.randint(40, 60), y + rng.randint(40, 60)], fill=(210, 40, 40))
    return img


Xs, Ys = [], []
for i in range(N_TRAIN + N_VAL):
    h, s, _ = teacher_pair(perturbed(i))
    Xs.append(h); Ys.append(s)
    if (i + 1) % 6 == 0:
        print("collected", i + 1)

X = torch.stack(Xs); Y = torch.stack(Ys)
Xtr, Ytr, Xva, Yva = X[:N_TRAIN], Y[:N_TRAIN], X[N_TRAIN:], Y[N_TRAIN:]
print("train", tuple(Xtr.shape), "val", tuple(Xva.shape))

In [ ]:
class Probe(nn.Module):
    def __init__(self, d, hid=HID):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d, hid), nn.GELU(), nn.Linear(hid, 1))

    def forward(self, x):
        return self.net(x).squeeze(-1)          # [B, |V|]


d = Xtr.shape[-1]
probe = Probe(d).cuda()
opt = torch.optim.Adam(probe.parameters(), lr=1e-3)
lossf = nn.BCEWithLogitsLoss()

for step in range(301):
    idx = torch.randint(0, Xtr.shape[0], (8,))
    xb = Xtr[idx].cuda()
    yb = Ytr[idx].cuda()
    logits = probe(xb)
    loss = lossf(logits, yb.clamp(0, 1))
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 100 == 0:
        with torch.no_grad():
            pv = torch.sigmoid(probe(Xva.cuda()))
            mae = (pv - Yva.cuda()).abs().mean().item()
        print(f"step {step:3d} loss {loss.item():.4f} val_MAE {mae:.4f}")

In [ ]:
# 在验证集的一个样本上看：探针图 vs 教师图
with torch.no_grad():
    pv = torch.sigmoid(probe(Xva[:1].cuda()))[0].cpu()

fig, ax = plt.subplots(1, 3, figsize=(12, 4))
overlay(image, Yva[0], "teacher (ILVAD)", ax[0])
overlay(image, pv, "student (probe)", ax[1])
ax[2].scatter(Yva[0].numpy(), pv.numpy(), s=6)
ax[2].set_xlabel("teacher"); ax[2].set_ylabel("student")
ax[2].set_title(f"corr = {np.corrcoef(Yva[0].numpy(), pv.numpy())[0, 1]:.3f}")
plt.tight_layout(); plt.show()

## 9. AOPC：证据图到底 faithful 不 faithful

按图排序，逐步「遮掉」top-k 视觉 token，看正确答案的 margin 掉得多快。

```
AOPC = 1/(K+1) · Σ_k [ margin(x) − margin(x \ Top_k(Ŝ)) ]
margin = logit(yes) − logit(no)
```

随机基线的期望是 `(k/|V|)·margin`，所以**只有显著高于随机基线才算有效**。

In [ ]:
def margin_with_map(model, inputs, score=None, mode="mask", alpha=0.0):
    set_map(score, alpha=alpha, mode=mode)
    y, n = yes_no_logits(model, inputs, processor)
    set_map(None)
    return y - n


def aopc(model, inputs, score, ks=(1, 2, 4, 8, 16), mode="mask"):
    base = margin_with_map(model, inputs, None)
    vals = []
    for k in ks:
        msk = torch.zeros_like(score)
        msk[score.topk(k).indices] = 1.0
        vals.append(base - margin_with_map(model, inputs, msk, mode=mode))
    return float(np.mean(vals)), vals


if PATCH_OK:
    base_margin = margin_with_map(model, inputs, None)
    rand = torch.rand_like(m["Shat"])
    a_ilvad, _ = aopc(model, inputs, m["Shat"])
    a_rand, _ = aopc(model, inputs, rand)
    print(f"margin(base) = {base_margin:.3f}")
    print(f"AOPC  ILVAD = {a_ilvad:.4f}")
    print(f"AOPC  random= {a_rand:.4f}")
    print(f"倍数 = {a_ilvad / (abs(a_rand) + 1e-6):.2f}x")
else:
    print("跳过：补丁自检未通过")

## 10. 已知问题与下一步

### 已知问题

1. **4bit 数值漂移**：与论文 fp16 的绝对数字不可比，只比差值。
2. **慢**：`use_cache=False` 便于教学与调试，但生成长文本时会 O(n²)。要快就写 KV cache 版本。
3. **`temperature` / 采样**：这里全部用贪心，和论文一致；不要混用采样。
4. **POPE 数据集 id 可能变**：`lmms-lab/POPE` 若失效，请换成当前可用的镜像并核对列名。
5. **自检 FAIL 的常见原因**：`transformers` 版本不匹配（本 notebook 按 4.45 写）、
   模型不是 Llama 系（Qwen2-VL 的 attention 是 GQA + 不同 RoPE 接口，需要另写补丁）。

### 下一步（按性价比）

- [ ] 把 `window_T`、`tau`、`alpha` 扫一遍，看证据图与 AOPC 怎么变（论文表 5 / 8 的微型版）。
- [ ] 加一个「随机图 / sink 图」对照，确认收益来自证据而不是「整体加大视觉权重」。
- [ ] 把探针从「模仿教师」换成「端到端」：冻结 LVLM，只用下游 margin 反传（4.3 的误差界说这更稳）。
- [ ] 换 Qwen2-VL-2B 验证跨架构可迁移性。

### 引用

```bibtex
@inproceedings{xie2026ilvad,
  title  = {Finding the Correct Visual Evidence Without Forgetting: Mitigating Hallucination in LVLMs via Inter-Layer Visual Attention Discrepancy},
  author = {Xie, Yutong and Hua, Zhenglin and Wang, Ran and Ng, Wing W. Y. and Wang, Xizhao and Jia, Yuheng},
  booktitle = {ICML},
  year   = {2026}
}
```